In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Charger le dataset MNIST (70 000 images de chiffres manuscrits)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("Forme des données d'entraînement :", X_train.shape)
print("Forme des données de test :", X_test.shape)
print("\nExemples de chiffres dans y_train :", y_train[:10])


In [ ]:
# Afficher les 10 premières images
plt.figure(figsize=(12, 4))
for i in range(10):
  plt.subplot(2, 5, i + 1)
  plt.imshow(X_train[i], cmap="gray")
  plt.title(f"Chiffre : {y_train[i]}")
  plt.axis("off")
  plt.tight_layout()
  plt.show()

In [ ]:
# 1. Normaliser les pixels (0-255 → 0-1)
X_train = X_train / 255.0
X_test = X_test / 255.0

# 2. Convertir les étiquettes en one-hot encoding
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print("✅ Données préparées !")
print(f"X_train min : {X_train.min()}, max : {X_train.max()}")
print(f"\nExemple de y_train[0] (chiffre {y_train[0]}) en one-hot :")
print(y_train_cat[0])

In [ ]:
from tensorflow.keras import layers

# Créer le CNN
modele = keras.Sequential([
    # Couche d'entrée : adapter la forme des images (28x28 → 28x28x1)
    layers.Reshape((28, 28, 1), input_shape=(28, 28)),

    # 1ère couche convolutive : 32 filtres de 3x3
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # 2ème couche convolutive : 64 filtres
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Aplatir pour passer aux couches denses
    layers.Flatten(),

    # Couches de classification
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")  # 10 sorties (un par chiffre)
])

# Compiler le modèle
modele.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Afficher l'architecture
modele.summary()

In [ ]:
# Entrainer le modele
historique = modele.fit(
    X_train, y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print("\✅ Entrainement terminé !")

In [ ]:
# Evaluer sur les données de test
loss_test, precision_test = modele.evaluate(X_test, y_test_cat, verbose=0)

print(f"✅ Précision finale sur le test : {precision_test * 100:.2f}%")
print(f"📉 Loss : {loss_test:.4f}")

In [ ]:
import numpy as np

# Faire des prédictions sur les 10 premières images de test
predictions = modele.predict(X_test[:10])
predictions_chiffres = np.argmax(predictions, axis=1)

# Afficher avec les vraies réponses
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i], cmap="gray")

    vrai = y_test[i]
    predit = predictions_chiffres[i]
    confiance = predictions[i][predit] * 100

    couleur = "green" if vrai == predit else "red"
    plt.title(f"Vrai: {vrai} | Prédit: {predit}\n({confiance:.1f}%)", color=couleur)
    plt.axis("off")

plt.tight_layout()
plt.show()